# Vanishing and exploding gradients
A 10-hidden-layer sigmoid network on make_moons: tiny first-layer gradients, a loss stuck at 0.69. Then the same
with 3 hidden layers and with ReLU. Last, a deep network whose gradients explode.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
import numpy as np
import pandas as pd
import keras
import tensorflow as tf
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

tf.config.experimental.enable_op_determinism()   # same numbers on every run

## Small numbers multiplied together

In [ ]:
print("0.1 ** 10 =", 0.1 ** 10)
print("sigmoid derivative at most 0.25; 0.25 ** 10 =", 0.25 ** 10)
print("1.5 ** 10 =", 1.5 ** 10)

## The data

In [ ]:
X, y = make_moons(n_samples=250, noise=0.05, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X.shape, np.bincount(y))

## Three networks

In [ ]:
def build(n_hidden, activation, seed=0):
    keras.utils.set_random_seed(seed)
    layers = [keras.Input(shape=(2,))]
    layers += [keras.layers.Dense(10, activation=activation) for _ in range(n_hidden)]
    layers += [keras.layers.Dense(1, activation="sigmoid")]
    model = keras.Sequential(layers)
    # plain SGD: every update is exactly learning rate x gradient
    model.compile(optimizer=keras.optimizers.SGD(learning_rate=0.5), loss="binary_crossentropy",
                  metrics=["accuracy"])
    return model

def layer_gradients(model):
    # mean |dL/dW| of each layer's weight matrix, on the training set, at the current weights
    with tf.GradientTape() as tape:
        loss = keras.losses.binary_crossentropy(y_train.reshape(-1, 1).astype("float32"),
                                                model(X_train.astype("float32")))
        loss = tf.reduce_mean(loss)
    kernels = [l.kernel for l in model.layers]
    return [float(tf.reduce_mean(tf.abs(g))) for g in tape.gradient(loss, kernels)]

deep_sigmoid = build(10, "sigmoid")
deep_sigmoid.summary()
g_sig = layer_gradients(deep_sigmoid)
print("mean |gradient| per layer, input side first:\n", np.array(g_sig).round(8))

## One epoch: how much do the first-layer weights change?

In [ ]:
old = deep_sigmoid.get_weights()[0].copy()               # 2 x 10 = 20 weights of the first layer
deep_sigmoid.fit(X_train, y_train, epochs=1, verbose=0)
new = deep_sigmoid.get_weights()[0]
print("old:", old[0, :5].round(6))
print("new:", new[0, :5].round(6))
print("largest change of the 20 weights:", np.abs(new - old).max())
# with plain SGD and one batch, (old - new) / lr would be the gradient itself; here 7 batches add up
print("(old - new) / lr, first five:", ((old - new) / 0.5)[0, :5])

## 100 epochs: deep sigmoid, shallow sigmoid, deep ReLU

In [ ]:
histories, grads = {}, {}
for name, n_hidden, act in [("10 sigmoid layers", 10, "sigmoid"), ("3 sigmoid layers", 3, "sigmoid"),
                            ("10 ReLU layers", 10, "relu")]:
    m = build(n_hidden, act)
    grads[name] = layer_gradients(m)
    w0 = m.get_weights()[0].copy()
    h = m.fit(X_train, y_train, epochs=100, verbose=0)
    histories[name] = h.history["loss"]
    acc = m.evaluate(X_test, y_test, verbose=0)[1]
    change = np.mean(np.abs(m.get_weights()[0] - w0))
    print(f"{name}: loss {h.history['loss'][0]:.3f} -> {h.history['loss'][-1]:.4f}, test accuracy {acc:.2f}, "
          f"mean first-layer change {change:.2e}")
pd.DataFrame(histories).rename_axis("epoch").to_csv("data/loss_histories.csv")

In [ ]:
rows = []
for name, g in grads.items():
    for i, v in enumerate(g):
        rows.append(dict(network=name, layer=i + 1, mean_abs_grad=v))
pd.DataFrame(rows).to_csv("data/layer_gradients.csv", index=False)
for name, g in grads.items():
    print(name, "first layer %.2e, last layer %.2e, ratio %.0f" % (g[0], g[-1], g[-1] / g[0]))

## Exploding gradients: large starting weights, no squashing activation
Each layer multiplies the backward signal by its weights. With weights drawn with standard deviation 1 and
10 nodes per layer, every layer makes it about sqrt(10) = 3 times bigger.

In [ ]:
def build_big(n_hidden, seed=0):
    keras.utils.set_random_seed(seed)
    init = keras.initializers.RandomNormal(stddev=1.0)
    layers = [keras.Input(shape=(2,))]
    layers += [keras.layers.Dense(10, activation="linear", kernel_initializer=init) for _ in range(n_hidden)]
    layers += [keras.layers.Dense(1, activation="sigmoid", kernel_initializer=init)]
    return keras.Sequential(layers)

big = build_big(10)
g_big = layer_gradients(big)
print("mean |gradient| per layer, input side first:\n", np.array(g_big).round(2))
pd.DataFrame(dict(layer=range(1, 12), mean_abs_grad=g_big)).to_csv("data/exploding_gradients.csv", index=False)
big.compile(optimizer=keras.optimizers.SGD(learning_rate=0.01), loss="binary_crossentropy")
h = big.fit(X_train, y_train, epochs=5, verbose=0)
print("loss per epoch with SGD, lr 0.01:", np.round(h.history["loss"], 3))

## Gradient clipping caps every gradient

In [ ]:
big2 = build_big(10)
big2.compile(optimizer=keras.optimizers.SGD(learning_rate=0.01, clipnorm=1.0), loss="binary_crossentropy")
h2 = big2.fit(X_train, y_train, epochs=5, verbose=0)
print("with clipnorm=1.0:", np.round(h2.history["loss"], 3))

## Do the spikes in the ReLU curve come from the learning rate?
The same 10-layer ReLU network with learning rates 0.5, 0.1 and 0.05; a spike is an epoch whose loss is at least 0.05 above the epoch before.

In [ ]:
for lr in [0.5, 0.1, 0.05]:
    m = build(10, "relu")
    m.optimizer.learning_rate.assign(lr)
    loss = np.array(m.fit(X_train, y_train, epochs=100, verbose=0).history["loss"])
    print(f"lr {lr}: final loss {loss[-1]:.4f}, spikes {(np.diff(loss) > 0.05).sum()}, test accuracy {m.evaluate(X_test, y_test, verbose=0)[1]:.2f}")
